# 28 — JIT Compilation Deep Dive

Understanding `jax.jit` — tracing, static args, compilation caching, and how to get the best out of XLA.

| Concept | What |
|---------|------|
| **Tracing** | JAX traces Python to build an XLA HLO graph |
| **Static args** | Values baked into the compiled graph |
| **Donation** | In-place buffer reuse via `donate_argnums` |
| **Lowering** | Inspect compiled HLO / MHLO |

In [ ]:
BACKEND = "cpu"

import sys, os, time
sys.path.insert(0, os.path.join(os.getcwd(), "..") if not os.getcwd().endswith("ql-jax") else ".")
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))

from notebooks._common import setup_backend, timed_ms
jax = setup_backend(BACKEND)
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from ql_jax.engines.analytic.black_scholes_merton import bsm_price
from ql_jax.engines.analytic.heston import heston_price
from ql_jax.engines.monte_carlo.european_mc import mc_european_bs

---
## 1. First-Call vs Subsequent Calls

In [ ]:
bsm_jit = jax.jit(bsm_price)

args = (100.0, 100.0, 1.0, 0.03, 0.01, 0.2, 1.0)

# First call: tracing + compilation + execution
t0 = time.perf_counter()
p1 = bsm_jit(*args)
t_first = (time.perf_counter() - t0) * 1000

# Second call: cached execution only
t0 = time.perf_counter()
for _ in range(100):
    p2 = bsm_jit(*args)
t_cached = (time.perf_counter() - t0) * 1000 / 100

print(f"First call (trace+compile+run) : {t_first:.2f} ms")
print(f"Cached call (run only)         : {t_cached:.4f} ms")
print(f"Ratio                          : {t_first / t_cached:.0f}×")

---
## 2. Recompilation: Shape & Type Changes

In [ ]:
batch_bsm = jax.jit(jax.vmap(bsm_price))

compile_times = []
exec_times = []
sizes = [10, 100, 1000, 10000]

for n in sizes:
    args_n = tuple(jnp.full(n, v) for v in [100.0, 100.0, 1.0, 0.03, 0.01, 0.2, 1.0])
    
    # First call → recompile for new shape
    t0 = time.perf_counter()
    _ = batch_bsm(*args_n)
    compile_times.append((time.perf_counter() - t0) * 1000)
    
    # Cached call
    t0 = time.perf_counter()
    for _ in range(10):
        _ = batch_bsm(*args_n)
    exec_times.append((time.perf_counter() - t0) * 1000 / 10)

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(sizes))
ax.bar(x - 0.2, compile_times, 0.35, label='First call (compile)', color='coral')
ax.bar(x + 0.2, exec_times, 0.35, label='Cached call', color='steelblue')
ax.set_xticks(x)
ax.set_xticklabels(sizes)
ax.set_xlabel('Batch Size')
ax.set_ylabel('Time (ms)')
ax.set_title('Compilation vs Execution Cost by Batch Size')
ax.legend()
ax.set_yscale('log')
plt.grid(True, alpha=0.3)
plt.show()

---
## 3. Static Args & Avoiding Recompilation

In [ ]:
# BAD: n_steps as a traced value → recompiles every time it changes
# GOOD: static_argnums tells JAX to bake it into the graph

def mc_pricer(S, K, T, r, q, sigma, n_paths, n_steps):
    price, _ = mc_european_bs(S, K, T, r, q, sigma, 1,
                              n_paths=n_paths, n_steps=n_steps)
    return price

# Mark n_paths and n_steps as static
mc_jit = jax.jit(mc_pricer, static_argnums=(6, 7))

# Same static args → no recompile
t0 = time.perf_counter()
_ = mc_jit(100.0, 100.0, 1.0, 0.03, 0.01, 0.2, 50_000, 100)
t_compile = (time.perf_counter() - t0) * 1000

# Change S → no recompile (traced arg)
t0 = time.perf_counter()
_ = mc_jit(105.0, 100.0, 1.0, 0.03, 0.01, 0.2, 50_000, 100)
t_same_static = (time.perf_counter() - t0) * 1000

# Change n_steps → recompile (static arg changed)
t0 = time.perf_counter()
_ = mc_jit(100.0, 100.0, 1.0, 0.03, 0.01, 0.2, 50_000, 200)
t_new_static = (time.perf_counter() - t0) * 1000

print(f"First compile                   : {t_compile:.1f} ms")
print(f"Same static args (cached)       : {t_same_static:.1f} ms")
print(f"Changed static arg (recompile)  : {t_new_static:.1f} ms")

---
## 4. Inspecting Compiled Code

In [ ]:
# Lower to HLO to see what XLA will actually compile
lowered = jax.jit(bsm_price).lower(100.0, 100.0, 1.0, 0.03, 0.01, 0.2, 1.0)
hlo = lowered.as_text()
print(f"HLO text length: {len(hlo)} chars")
print("First 500 chars:")
print(hlo[:500])

In [ ]:
# Cost analysis
compiled = lowered.compile()
print(f"Cost analysis available: {hasattr(compiled, 'cost_analysis')}")
try:
    ca = compiled.cost_analysis()
    if ca:
        for k, v in list(ca[0].items())[:5]:
            print(f"  {k}: {v}")
except Exception as e:
    print(f"  (cost analysis: {e})")

---
## 5. JIT + AD Interaction

In [ ]:
# JIT wrapping AD vs AD wrapping JIT — both work, but ordering affects compilation

# Option A: jit(grad(f))
grad_then_jit = jax.jit(jax.grad(bsm_price, argnums=0))

# Option B: grad(jit(f))
jit_then_grad = jax.grad(jax.jit(bsm_price), argnums=0)

# Warm up
args = (100.0, 100.0, 1.0, 0.03, 0.01, 0.2, 1.0)
_ = grad_then_jit(*args)
_ = jit_then_grad(*args)

ms_a, delta_a = timed_ms(grad_then_jit, *args)
ms_b, delta_b = timed_ms(jit_then_grad, *args)

print(f"jit(grad(f)) : {ms_a:.4f} ms  →  Δ = {float(delta_a):.8f}")
print(f"grad(jit(f)) : {ms_b:.4f} ms  →  Δ = {float(delta_b):.8f}")
print(f"Same result  : {jnp.allclose(delta_a, delta_b)}")

---
## 6. Benchmark Across Engines

In [ ]:
engines = {
    'BSM':    lambda: bsm_price(100.0, 100.0, 1.0, 0.03, 0.01, 0.2, 1.0),
    'Heston': lambda: heston_price(100.0, 100.0, 1.0, 0.03, 0.01, 0.04, 1.5, 0.04, 0.3, -0.7, 1),
}

results = {}
for name, fn in engines.items():
    fn_jit = jax.jit(fn)
    _ = fn_jit()  # warm
    ms_eager, _ = timed_ms(fn)
    ms_jit, _ = timed_ms(fn_jit)
    results[name] = (ms_eager, ms_jit)
    print(f"{name:8s}: eager {ms_eager:.3f} ms → JIT {ms_jit:.4f} ms  ({ms_eager/ms_jit:.0f}×)")

labels = list(results.keys())
eager_times = [results[k][0] for k in labels]
jit_times = [results[k][1] for k in labels]

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(labels))
ax.bar(x - 0.2, eager_times, 0.35, label='Eager', color='coral')
ax.bar(x + 0.2, jit_times, 0.35, label='JIT', color='steelblue')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('Time (ms)')
ax.set_title('Eager vs JIT Execution Time')
ax.legend()
ax.set_yscale('log')
plt.grid(True, alpha=0.3)
plt.show()

---
## 7. Exercises

1. **Donation**: Use `donate_argnums` to reuse an input buffer and measure the effect.
2. **Trace debugging**: Use `jax.make_jaxpr(bsm_price)(*args)` to inspect the JAX expression.
3. **lax.cond vs if/else**: Compare compilation behaviour when branching on a traced value.

---
*End of Notebook 28*